# M-2LRF Real Benchmark (GPT-2 on Colab GPU)
Runtime > Change runtime type > GPU (T4) then Run All.

In [1]:
!pip -q install transformers datasets accelerate

In [2]:
import math, time, torch, torch.nn as nn, torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from datasets import load_dataset
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [3]:
A0, A1 = 0.4527786409, 1.5104181947

class M2LRF2BitLinear(nn.Module):
    def __init__(self, in_f, out_f, rank=16, alpha=16.0, bias=True):
        super().__init__()
        self.in_f, self.out_f, self.rank = in_f, out_f, rank
        self.scaling = alpha/rank
        self.packed_k = math.ceil(in_f/4)
        self.register_buffer('packed', torch.zeros(out_f, self.packed_k, dtype=torch.uint8))
        self.register_buffer('a0', torch.zeros(out_f,1))
        self.register_buffer('a1', torch.zeros(out_f,1))
        self.lora_A = nn.Parameter(torch.zeros(rank, in_f))
        self.lora_B = nn.Parameter(torch.zeros(out_f, rank))
        self.bias = nn.Parameter(torch.zeros(out_f)) if bias else None

    @torch.no_grad()
    def init_from(self, W, b=None):
        w = W.float()
        std = w.std(dim=-1, keepdim=True).clamp_min(1e-6)
        a0, a1 = std*A0, std*A1
        thresh = (a0+a1)/2
        pos = w >= 0
        abs_w = w.abs()
        codes = torch.zeros_like(w, dtype=torch.uint8)
        codes[(~pos)&(abs_w>thresh)] = 0
        codes[(~pos)&(abs_w<=thresh)] = 1
        codes[pos&(abs_w<=thresh)] = 2
        codes[pos&(abs_w>thresh)] = 3
        pad = self.packed_k*4 - self.in_f
        if pad>0: codes = F.pad(codes,(0,pad))
        c = codes.view(self.out_f,-1,4)
        packed = (c[...,0]<<0)|(c[...,1]<<2)|(c[...,2]<<4)|(c[...,3]<<6)
        self.packed.copy_(packed.to(torch.uint8))
        self.a0.copy_(a0); self.a1.copy_(a1)
        if b is not None and self.bias is not None: self.bias.copy_(b)
        w_dq = self._dequant()
        resid = w - w_dq.float()
        try:
            u,s,v = torch.svd_lowrank(resid, q=self.rank, niter=4)
            ss = torch.sqrt(s.clamp_min(1e-8))
            nf = 1.0/math.sqrt(self.scaling)
            self.lora_B.copy_((u*ss)*nf)
            self.lora_A.copy_((ss.unsqueeze(1)*v.t())*nf)
        except Exception:
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def _dequant(self):
        p = self.packed
        c0=(p>>0)&3; c1=(p>>2)&3; c2=(p>>4)&3; c3=(p>>6)&3
        codes = torch.stack([c0,c1,c2,c3],dim=-1).flatten(1)[:, :self.in_f]
        w = torch.zeros(self.out_f, self.in_f, device=p.device)
        w = torch.where(codes==0, -self.a1, w)
        w = torch.where(codes==1, -self.a0, w)
        w = torch.where(codes==2, self.a0, w)
        w = torch.where(codes==3, self.a1, w)
        return w

    def forward(self, x):
        w = self._dequant().to(x.dtype)
        out = F.linear(x, w)
        lora = F.linear(F.linear(x.float(), self.lora_A), self.lora_B).to(x.dtype)*self.scaling
        out = out+lora
        if self.bias is not None: out = out + self.bias
        return out

## Sanity check: quantizer correctness

In [4]:
torch.manual_seed(0)
W = torch.randn(1024,1024)
layer = M2LRF2BitLinear(1024,1024,rank=16)
layer.init_from(W)
w_dq = layer._dequant()
mse = ((W-w_dq)**2).mean().item()
var = (W**2).mean().item()
print('Empirical SQNR (theory 9.30 dB):', 10*math.log10(var/mse), 'dB')

Empirical SQNR (theory 9.30 dB): 9.305924177701947 dB


## Real GPT-2 + WikiText-2 benchmark: FP16 baseline vs M-2LRF

In [6]:
tok = GPT2TokenizerFast.from_pretrained('gpt2')
ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train[:2%]')
text = '\n\n'.join([t for t in ds['text'] if t.strip()])[:200000]
ids = tok(text, return_tensors='pt').input_ids[0]
SEQ=128; BATCH=4
def get_batch(step):
    chunks=[]
    for i in range(BATCH):
        s=(step*BATCH+i)*SEQ % (len(ids)-SEQ-1)
        chunks.append(ids[s:s+SEQ+1])
    b = torch.stack(chunks)
    return b[:, :-1].to(device), b[:, 1:].to(device)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (43625 > 1024). Running this sequence through the model will result in indexing errors


In [7]:
def replace_linears_with_m2lrf(model, rank=16):
    for block in model.transformer.h:
        for name in ['c_attn','c_proj']:
            conv1d = getattr(block.attn if name=='c_attn' else block.attn, name, None)
        # GPT2 uses Conv1D not Linear; convert weight shape (in,out)->(out,in)
    return model  # placeholder if you want deeper surgery; MLP swap done below

def swap_mlp(model, rank=16):
    layers=[]
    for block in model.transformer.h:
        w_fc = block.mlp.c_fc.weight.data.t().contiguous()   # (out,in)
        b_fc = block.mlp.c_fc.bias.data.clone()
        new_fc = M2LRF2BitLinear(w_fc.shape[1], w_fc.shape[0], rank=rank).to(device)
        new_fc.init_from(w_fc.to(device), b_fc.to(device))
        block.mlp.c_fc = new_fc
        layers.append(new_fc)

        w_proj = block.mlp.c_proj.weight.data.t().contiguous()
        b_proj = block.mlp.c_proj.bias.data.clone()
        new_proj = M2LRF2BitLinear(w_proj.shape[1], w_proj.shape[0], rank=rank).to(device)
        new_proj.init_from(w_proj.to(device), b_proj.to(device))
        block.mlp.c_proj = new_proj
        layers.append(new_proj)

    # monkeypatch forward: Conv1D-style modules expect (*, in) -> (*, out) same as our Linear
    for block in model.transformer.h:
        orig_forward = block.mlp.forward
        def make_forward(m):
            def fwd(x):
                x = m.c_fc(x)
                x = m.act(x)
                x = m.c_proj(x)
                return x
            return fwd
        block.mlp.forward = make_forward(block.mlp)
    return layers

In [8]:
def run_bench(use_m2lrf, steps=40, lr=1e-4):
    torch.manual_seed(0)
    model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
    trainable = []
    if use_m2lrf:
        layers = swap_mlp(model, rank=16)
        for p in model.parameters(): p.requires_grad_(False)
        for l in layers:
            l.lora_A.requires_grad_(True); l.lora_B.requires_grad_(True)
            trainable += [l.lora_A, l.lora_B]
    else:
        trainable = list(model.parameters())

    opt = torch.optim.AdamW(trainable, lr=lr)
    torch.cuda.reset_peak_memory_stats() if device=='cuda' else None
    weight_mem = sum(b.numel()*b.element_size() for b in model.buffers()) + \
                 sum(p.numel()*p.element_size() for n,p in model.named_parameters() if 'lora' not in n)

    losses=[]
    t0=time.time()
    for step in range(steps+1):
        x,y = get_batch(step)
        out = model(input_ids=x, labels=y)
        loss = out.loss
        if step==0: step0_loss = loss.item()
        losses.append(loss.item())
        loss.backward()
        opt.step(); opt.zero_grad()
    elapsed = time.time()-t0
    peak_mem = torch.cuda.max_memory_allocated()/1e6 if device=='cuda' else 0

    return dict(step0_loss=step0_loss, stepN_loss=losses[-1], elapsed=elapsed,
                peak_mem_MB=peak_mem, weight_mem_MB=weight_mem/1e6)

print('Running FP16/FP32 baseline...')
baseline = run_bench(use_m2lrf=False)
print(baseline)

print('Running M-2LRF 2-bit + LoRA...')
m2lrf = run_bench(use_m2lrf=True)
print(m2lrf)

Running FP16/FP32 baseline...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'step0_loss': 8.897972106933594, 'stepN_loss': 6.679435729980469, 'elapsed': 6.750701189041138, 'peak_mem_MB': 2670.144, 'weight_mem_MB': 497.759232}
Running M-2LRF 2-bit + LoRA...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

{'step0_loss': 9.085660934448242, 'stepN_loss': 7.487401485443115, 'elapsed': 4.11248517036438, 'peak_mem_MB': 1512.068096, 'weight_mem_MB': 285.791232}


In [9]:
import pandas as pd
df = pd.DataFrame([baseline, m2lrf], index=['Baseline (full FP32)','M-2LRF 2-bit+LoRA r=16'])
df

,step0_loss,stepN_loss,elapsed,peak_mem_MB,weight_mem_MB
Baseline (full FP32),8.897972,6.679436,6.750701,2670.144000,497.759232
M-2LRF 2-bit+LoRA r=16,9.085661,7.487401,4.112485,1512.068096,285.791232


**Notes:**
- This only quantizes the MLP `c_fc`/`c_proj` blocks (attention left untouched) for simplicity — real weight_mem numbers will differ from the paper's full-model claim, adjust if you want attention swapped too.
- `step0_loss`/`stepN_loss`/`elapsed`/`peak_mem_MB` here are REAL measured numbers from this run — copy them into the paper's Section 8.1 table instead of the placeholder figures, and note the GPU type Colab assigned you (check via `!nvidia-smi`).

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader